# Track 2d: Routed models, one per day type (Eshrit)

**Idea.** First decide what kind of day it is, then use a model built for that day type. The day definition
lives in its own module, `day_types.py`, so it can be changed or reused by other tracks.

| Day type (priority order) | Training days, 2024 fold |
|---|---|
| Russell reconstitution | 3 |
| MSCI review | 12 |
| S&P quarterly rebalance / quad witching | 12 |
| Month-end (other) | ~25 |
| Monthly options expiry | ~24 |
| Normal | ~670 |

**Two versions, same day types, same features** (the `track2_gmm_probs_v1` features: incumbent 73 +
GMM regime probabilities + index-event flags):

| Variant | How each day type's model learns |
|---|---|
| `track2_routed_hard_v1` | **Fully separate.** Each day type gets its own ridge, trained only on its own days. The normal-day model trains on normal days only. |
| `track2_routed_shrunk_v1` | **Specialists that start from the general model.** One general model learns from all days (identical to `track2_gmm_probs_v1`). Each event-day specialist learns only the *difference* from it, and a penalty pulls that difference toward zero. Few days means it stays close to the general model. Normal days use the general model. |

**Hypothesis.** The fully separate version overfits the rare day types (3 Russell days). The shrunk version
improves event days without touching normal days.

**Protocol (no leakage, same for both):**
- Event-day models train on *all* stocks on past event days, from the full panel (about 3× the rows in the
  training sample), but only on dates before the validation window (tuning) or before the test year (final fit).
- One penalty strength shared by all event types, plus, for the separate version, one for the normal model.
  Both are chosen on the validation window: validation event days for the event models, validation normal-day
  rows for the normal model.
- A day type with no training days yet falls back to the general model.
- Both versions use the general model's preprocessing, so the only difference between them is how the
  coefficients are learned.

In [1]:
import sys
from pathlib import Path
from functools import lru_cache
import numpy as np, pandas as pd
from sklearn.mixture import GaussianMixture
sys.path.insert(0, str(Path.cwd()))
from intraday_harness import Harness, leaderboard, RIDGE_GRID, REBALANCE, SEED, OUT, INCUMBENT, ANCHOR, DEVELOPMENT_YEARS
from notebooks.modeling_utils import fit_ridge, predict_ridge, forecast_metrics
from day_types import DAY_TYPES, EVENT_TYPES, index_calendar, day_type

OWNER = "Eshrit"
QUICK = False                     # True = 2024 only, not recorded
K_GRID = (2, 3, 4)
EVENT_GRID = (1e-4, 1e-3, 1e-2, 3e-2, 0.1, 0.3, 1.0, 3.0, 10.0)   # penalty for event-day models
h = Harness(quick=QUICK)
calendar = index_calendar(h.con)

## 1. General-model features (same as `track2_gmm_probs_v1`)

In [2]:
cols = ["date", "permno", "cp_available", "cp_volume_surprise20", "cp_trade_count_surprise20",
        "cp_recent_30m_volume_share", "cp_vwap_return_prior_close"]
panel = h.full_panel("2021-01-01", "2026-01-01", cols)
live = panel[panel.cp_available.astype(bool)].copy()
live["log_surprise"] = np.log(live.cp_volume_surprise20.clip(lower=1e-3))
live["log_trade_surprise"] = np.log(live.cp_trade_count_surprise20.clip(lower=1e-3))
live["abs_move"] = live.cp_vwap_return_prior_close.abs()
live["hot"] = live.log_surprise > np.log(1.5)
g = live.groupby("date")
state = pd.DataFrame({
    "mkt_pace": g.log_surprise.median(),
    "mkt_trade_pace": g.log_trade_surprise.median(),
    "mkt_dispersion": g.log_surprise.quantile(0.75) - g.log_surprise.quantile(0.25),
    "mkt_breadth": g.hot.mean(),
    "mkt_recent30": g.cp_recent_30m_volume_share.median(),
    "mkt_abs_move": g.abs_move.median(),
})
STATE_COLS = list(state.columns)
events = calendar.rename(columns={"russell": "mkt_russell_recon", "msci": "mkt_msci_review", "sp_rebal": "mkt_sp_rebal"})
EVENT_COLS = ["mkt_russell_recon", "mkt_msci_review", "mkt_sp_rebal"]

@lru_cache(maxsize=None)
def regimes(train_end: str, k: int) -> pd.DataFrame:
    train = state.loc[state.index < pd.Timestamp(train_end), STATE_COLS]
    mu, sd = train.mean(), train.std()
    gmm = GaussianMixture(k, covariance_type="full", reg_covar=1e-3, n_init=5, random_state=SEED)
    gmm.fit(((train - mu) / sd).to_numpy())
    p = gmm.predict_proba(((state[STATE_COLS] - mu) / sd).to_numpy())
    order = np.argsort(gmm.means_[:, STATE_COLS.index("mkt_pace")])
    return pd.DataFrame(p[:, order], index=state.index, columns=[f"regime{k}_p{j}" for j in range(k)])

def valid_start(year):
    return "2021-10-01" if year == 2022 else f"{year - 1}-01-01"

def regime_features(frame, fold_year, role):
    train_end = valid_start(fold_year) if role in ("tune", "valid") else f"{fold_year}-01-01"
    out = events[EVENT_COLS].reindex(frame.date).fillna(0).set_axis(frame.index)
    for k in K_GRID:
        out = out.join(regimes(train_end, k).reindex(frame.date).set_axis(frame.index))
    return out

h.add_features(regime_features)
def extra(k): return [f"regime{k}_p{j}" for j in range(k)] + EVENT_COLS

## 2. Day types and event-day rows from the full panel

The training sample has only a few hundred rows per day, so event-day models load *every* stock on past event
days instead.

In [3]:
all_dates = pd.DataFrame({"date": state.index})
flags = h.full_panel("2021-01-01", "2026-01-01", ["date", "is_last_trading_day_month", "quarterly_options_expiry",
                                                  "monthly_options_expiry"]).drop_duplicates("date")
date_type = pd.Series(day_type(flags, calendar).to_numpy(), index=flags.date, name="day_type")
counts = date_type.groupby([date_type, date_type.index.year]).size().unstack(fill_value=0)
display(counts.reindex(DAY_TYPES))

def load_event_rows(start, end, fold_year, role):
    dates = date_type[(date_type != "normal") & (date_type.index >= pd.Timestamp(start)) & (date_type.index < pd.Timestamp(end))].index
    if len(dates) == 0:
        return pd.DataFrame()
    listing = ", ".join(f"DATE '{d:%Y-%m-%d}'" for d in dates)
    frame = h.con.sql(f'''SELECT {", ".join(h._columns)} FROM comparison_panel
                          WHERE cp_available AND date IN ({listing}) ORDER BY date, permno''').df()
    frame["date"] = pd.to_datetime(frame.date)
    frame["anchor_forecast"] = frame[ANCHOR].astype(float)
    frame["log_daily_forecast"] = np.log1p(frame.anchor_forecast)
    for function in h._feature_functions:
        frame = pd.concat([frame, function(frame, fold_year, role).set_axis(frame.index)], axis=1)
    frame["day_type"] = date_type.reindex(frame.date).to_numpy()
    return frame

date,2021,2022,2023,2024,2025
day_type,,,,,
russell,1,1,1,1,1
msci,4,4,4,4,4
sp_quad,4,4,4,4,4
month_end,9,8,8,7,9
monthly_expiry,8,8,8,8,8
normal,225,225,223,225,221


## 3. Fit both versions fold by fold

In [4]:
def log_rmse(fold, frame, correction, bounds):
    forecast = fold.to_shares(correction, frame, bounds)
    return float(np.sqrt(np.mean((np.log1p(forecast) - frame.log1p_auction_volume.to_numpy(float)) ** 2)))

def fit_general(fold, fit_frame, k, strength):
    prep = fold.preprocessor(fit_frame, extra_continuous=extra(k))
    return prep, fit_ridge(prep.transform(fit_frame), fold.target(fit_frame), strength)

def event_models(prep, rows, beta_general, strength, shrink):
    # One model per event type. shrink=True learns the difference from the general model.
    models = {}
    for t in EVENT_TYPES:
        part = rows[rows.day_type == t] if len(rows) else rows
        if len(part) == 0:
            continue
        x, z = prep.transform(part), Fold_target(part)
        if shrink:
            models[t] = beta_general + fit_ridge(x, z - predict_ridge(x, beta_general), strength)
        else:
            models[t] = fit_ridge(x, z, strength)
    return models

def Fold_target(frame):
    return (frame.log1p_auction_volume - frame.log_daily_forecast).to_numpy(float)

def route(prep, frame, types, models, default_beta):
    # Prediction per row from the model of its day type (default model for normal days and unseen types).
    x = prep.transform(frame)
    out = predict_ridge(x, default_beta)
    for t, beta in models.items():
        mask = (types == t).to_numpy()
        if mask.any():
            out[mask] = predict_ridge(x[mask], beta)
    return out

settings, general_check = [], []
for fold in h.folds():
    year, vs, ys = fold.year, valid_start(fold.year), f"{fold.year}-01-01"
    bounds_tune = fold.bounds(fold.tune)

    # --- general model: K and strength on validation (identical to track2_gmm_probs_v1)
    best = None
    for k in K_GRID:
        prep, _ = fit_general(fold, fold.tune, k, 1.0)
        x_tune, x_valid = prep.transform(fold.tune), prep.transform(fold.valid)
        for s in RIDGE_GRID:
            beta = fit_ridge(x_tune, fold.target(fold.tune), s)
            r = log_rmse(fold, fold.valid, predict_ridge(x_valid, beta), bounds_tune)
            best = min(best or (r, k, s), (r, k, s))
    _, k, s_general = best
    prep_t, beta_t = fit_general(fold, fold.tune, k, s_general)

    # --- tuning-stage event rows: train before the validation window, score on validation event days
    ev_train = load_event_rows("2021-01-01", vs, year, "tune")
    ev_valid = load_event_rows(vs, ys, year, "valid")
    choice = {}
    for shrink in (True, False):
        trials = []
        for s in EVENT_GRID:
            models = event_models(prep_t, ev_train, beta_t, s, shrink)
            pred = route(prep_t, ev_valid, ev_valid.day_type, models, beta_t)
            trials.append((log_rmse(fold, ev_valid, pred, bounds_tune), s))
        choice[shrink] = min(trials)
    # normal-day model for the fully separate version: trained on normal rows only
    tune_types, valid_types = day_type(fold.tune, calendar), day_type(fold.valid, calendar)
    tn, vn = fold.tune[tune_types == "normal"], fold.valid[valid_types == "normal"]
    x_tn, x_vn = prep_t.transform(tn), prep_t.transform(vn)
    normal_choice = min((log_rmse(fold, vn, predict_ridge(x_vn, fit_ridge(x_tn, fold.target(tn), s)), bounds_tune), s)
                        for s in RIDGE_GRID)

    # --- final fit through the year before the test year
    prep, beta_general = fit_general(fold, fold.refit, k, s_general)
    ev_refit = load_event_rows("2021-01-01", ys, year, "refit")
    test_types = day_type(fold.test, calendar)
    refit_types = day_type(fold.refit, calendar)
    normal_rows = fold.refit[refit_types == "normal"]
    beta_normal = fit_ridge(prep.transform(normal_rows), fold.target(normal_rows), normal_choice[1])

    shrunk = event_models(prep, ev_refit, beta_general, choice[True][1], shrink=True)
    hard = event_models(prep, ev_refit, beta_general, choice[False][1], shrink=False)
    h.submit(fold, route(prep, fold.test, test_types, shrunk, beta_general), name="track2_routed_shrunk_v1", owner=OWNER,
             description="track2_gmm_probs_v1 general model + one event-day specialist per day type that learns only the difference from it (shrunk toward it); day types from day_types.py")
    h.submit(fold, route(prep, fold.test, test_types, hard, beta_normal), name="track2_routed_hard_v1", owner=OWNER,
             description="Fully separate ridge per day type (Russell, MSCI, S&P/quad, month-end, monthly expiry, normal) on track2_gmm_probs_v1 features; day types from day_types.py")
    h._submissions.setdefault("_general", []).append(h._prediction_frame(fold, fold.to_shares(predict_ridge(prep.transform(fold.test), beta_general))))
    settings.append({"year": year, "k": k, "general_strength": s_general,
                     "shrunk_event_strength": choice[True][1], "hard_event_strength": choice[False][1],
                     "hard_normal_strength": normal_choice[1],
                     "event_train_days": ev_refit.date.nunique(), "event_train_rows": len(ev_refit)})
pd.DataFrame(settings)

track2_routed_shrunk_v1 2022: log RMSE 0.49348, WAPE 35.5574%, bias -12.26%


track2_routed_hard_v1 2022: log RMSE 0.49561, WAPE 35.7432%, bias -12.20%


track2_routed_shrunk_v1 2023: log RMSE 0.44265, WAPE 33.6065%, bias -11.11%


track2_routed_hard_v1 2023: log RMSE 0.44279, WAPE 33.7139%, bias -11.42%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

track2_routed_shrunk_v1 2024: log RMSE 0.41500, WAPE 32.2658%, bias -11.74%


track2_routed_hard_v1 2024: log RMSE 0.41517, WAPE 32.2631%, bias -12.33%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

track2_routed_shrunk_v1 2025: log RMSE 0.40130, WAPE 31.3956%, bias -4.72%


track2_routed_hard_v1 2025: log RMSE 0.40063, WAPE 31.3435%, bias -4.68%


,year,k,general_strength,shrunk_event_strength,hard_event_strength,hard_normal_strength,event_train_days,event_train_rows
0,2022,4,0.0100,1.0,1.000,0.000001,26,33568
1,2023,2,0.0100,1.0,1.000,0.001000,51,66406
2,2024,4,0.0100,3.0,1.000,0.001000,76,98338
3,2025,2,0.0001,10.0,0.001,0.000100,100,128519


**Check:** the general model inside this notebook must reproduce `track2_gmm_probs_v1` exactly.

In [5]:
saved = pd.read_parquet(OUT / "submissions" / "track2_gmm_probs_v1" / "predictions.parquet")
general = pd.concat(h._submissions.pop("_general"))
both = general.merge(saved, on=["date", "permno"], suffixes=("", "_saved"))
diff = (np.log1p(both.forecast) - np.log1p(both.forecast_saved)).abs().max()
print(f"rows {len(both):,}, max log difference {diff:.2e}", "PASS" if diff < 1e-6 else "FAIL")

rows 1,260,168, max log difference 0.00e+00 PASS


## 4. Results

In [6]:
VARIANTS = ["track2_routed_hard_v1", "track2_routed_shrunk_v1"]
results = {name: h.report(name) for name in VARIANTS}
pd.concat({n: r["summary"].loc[[INCUMBENT, n]] for n, r in results.items()}).round(5)

development_rmse_log  \
                        model                                           
track2_routed_hard_v1   incumbent_tree_ridge                  0.45445   
                        track2_routed_hard_v1                 0.45119   
track2_routed_shrunk_v1 incumbent_tree_ridge                  0.45445   
                        track2_routed_shrunk_v1               0.45038   

                                                 development_wape  \
                        model                                       
track2_routed_hard_v1   incumbent_tree_ridge              0.33838   
                        track2_routed_hard_v1             0.33907   
track2_routed_shrunk_v1 incumbent_tree_ridge              0.33838   
                        track2_routed_shrunk_v1           0.33810   

                                                 development_bias  \
                        model                                       
track2_routed_hard_v1   incumbent_tree_ridge             -0.13937   
                        track2_routed_hard_v1            -0.11980   
track2_routed_shrunk_v1 incumbent_tree_ridge             -0.13937   
                        track2_routed_shrunk_v1          -0.11702   

                                                 confirmation_rmse_log  \
                        model                                            
track2_routed_hard_v1   incumbent_tree_ridge                   0.40613   
                        track2_routed_hard_v1                  0.40063   
track2_routed_shrunk_v1 incumbent_tree_ridge                   0.40613   
                        track2_routed_shrunk_v1                0.40130   

                                                 confirmation_wape  
                        model                                       
track2_routed_hard_v1   incumbent_tree_ridge               0.31597  
                        track2_routed_hard_v1              0.31343  
track2_routed_shrunk_v1 incumbent_tree_ridge               0.31597  
                        track2_routed_shrunk_v1            0.31396

In [7]:
pd.DataFrame({n: r["gate"] for n, r in results.items()}) if not QUICK else "Gate is only applied on the full 2022-2025 run."

,track2_routed_hard_v1,track2_routed_shrunk_v1
dev_rmse_gain,0.003264,0.004074
gain_ok,True,True
worst_year_change,0.000419,-0.001705
annual_ok,True,True
wape_ok,False,True
interval_ok,True,True
passes_gate,False,True


### Against the general model (`track2_gmm_probs_v1`): paired differences, negative means better

In [8]:
reference = saved[saved.date.dt.year.isin(h.years)][["date", "permno", "forecast"]].rename(columns={"forecast": "ref"})
rows = []
for name in VARIANTS:
    cand = pd.concat(h._submissions[name])[["date", "permno", "auction_volume", "forecast"]].rename(columns={"forecast": "cand"})
    paired = cand.merge(reference, on=["date", "permno"], validate="one_to_one")
    rows.append(h._paired_intervals(paired, "ref", "cand").query("metric in ['rmse_log', 'wape']").assign(variant=name))
pd.concat(rows)[["variant", "period", "metric", "difference", "lower", "upper"]].round(5)

,variant,period,metric,difference,lower,upper
0,track2_routed_hard_v1,development,rmse_log,-0.00042,-0.00227,0.00112
1,track2_routed_hard_v1,development,wape,0.00292,0.00000,0.00588
4,track2_routed_hard_v1,confirmation,rmse_log,-0.00339,-0.01216,0.00257
5,track2_routed_hard_v1,confirmation,wape,-0.00044,-0.00923,0.00737
0,track2_routed_shrunk_v1,development,rmse_log,-0.00123,-0.00285,0.00006
1,track2_routed_shrunk_v1,development,wape,0.00195,-0.00055,0.00455
4,track2_routed_shrunk_v1,confirmation,rmse_log,-0.00273,-0.00773,0.00007
5,track2_routed_shrunk_v1,confirmation,wape,0.00008,-0.00258,0.00223


### By day type (development years 2022–2024, log RMSE, lower is better)

In [9]:
frame = pd.concat(h._submissions["track2_routed_hard_v1"])[["date", "permno", "auction_volume", "forecast"]].rename(columns={"forecast": "hard"})
frame = frame.merge(pd.concat(h._submissions["track2_routed_shrunk_v1"])[["date", "permno", "forecast"]].rename(columns={"forecast": "shrunk"}), on=["date", "permno"])
frame = frame.merge(saved[["date", "permno", "forecast"]].rename(columns={"forecast": "general"}), on=["date", "permno"])
frame = frame.merge(h._incumbent_predictions()[["date", "permno", "forecast"]].rename(columns={"forecast": "incumbent"}), on=["date", "permno"])
frame = frame[frame.date.dt.year.isin(DEVELOPMENT_YEARS)]
frame["day_type"] = date_type.reindex(frame.date).to_numpy()
rows = []
for t, part in frame.groupby("day_type"):
    row = {"day_type": t, "days": part.date.nunique(), "rows": len(part)}
    for m in ("incumbent", "general", "hard", "shrunk"):
        row[m] = forecast_metrics(part.auction_volume, part[m])["rmse_log"]
    rows.append(row)
by_type = pd.DataFrame(rows).set_index("day_type").reindex(DAY_TYPES)
by_type["hard vs general"] = by_type.hard - by_type.general
by_type["shrunk vs general"] = by_type.shrunk - by_type.general
by_type.round(4)

,days,rows,incumbent,general,hard,shrunk,hard vs general,shrunk vs general
day_type,,,,,,,,
russell,3,3853,1.0112,1.0194,0.9384,0.9357,-0.0810,-0.0838
msci,12,15404,0.7734,0.7149,0.7111,0.6961,-0.0037,-0.0188
sp_quad,12,15431,0.7308,0.7314,0.7408,0.7283,0.0094,-0.0032
month_end,23,29586,0.5801,0.5733,0.5825,0.5759,0.0092,0.0026
monthly_expiry,24,30713,0.5300,0.5285,0.5361,0.5294,0.0077,0.0009
normal,673,861009,0.4295,0.4284,0.4280,0.4284,-0.0004,0.0000


In [10]:
leaderboard()[["name", "development_rmse_log", "development_wape", "confirmation_rmse_log", "dev_rmse_gain", "passes_gate"]]

,name,development_rmse_log,development_wape,confirmation_rmse_log,dev_rmse_gain,passes_gate
0,track2_routed_shrunk_v1,0.450377,0.338099,0.401296,0.004074,True
1,track2_routed_hard_v1,0.451188,0.339067,0.400628,0.003264,False
2,track2_exposure_v1,0.451597,0.335962,0.403949,0.002854,True
3,track2_gmm_interact_v1,0.451602,0.336080,0.404027,0.002850,True
4,track2_gmm_probs_v1,0.451603,0.336145,0.404022,0.002848,True
5,track2_hmm_live_v1,0.451856,0.336376,0.403953,0.002595,True
6,track2_events_only_v1,0.452110,0.336405,0.403971,0.002341,True
7,track2_hmm_slow_v1,0.452421,0.336382,0.403975,0.002030,True
8,track2_error_momentum_v1,0.452538,0.336552,0.404680,0.001913,False
9,track2_event_window_v1,0.452690,0.337931,0.404938,0.001761,False


## Findings (full run, 2022–2025)

| Model | Dev log RMSE | Gain vs incumbent | Dev WAPE | 2025 log RMSE | Gate |
|---|---|---|---|---|---|
| Incumbent | 0.45445 | | 33.84% | 0.40613 | |
| General: `track2_gmm_probs_v1` | 0.45160 | 0.00285 | 33.61% | 0.40402 | pass |
| `track2_routed_hard_v1` (fully separate) | 0.45119 | 0.00326 | 33.91% | 0.40063 | **fail** (WAPE) |
| **`track2_routed_shrunk_v1` (specialists)** | **0.45038** | **0.00407** | 33.81% | 0.40130 | **pass** |

The general model inside this notebook reproduces `track2_gmm_probs_v1` exactly (max difference 0).

1. **Specialists that start from the general model are the best Track 2 model so far.**
   - Development is 0.00123 better than the general model, with an interval from −0.00285 to +0.00006,
     which just touches zero.
   - 2025 is 0.0027 better (interval −0.0077 to +0.00007).
   - Normal days are unchanged by design.
2. **They fix the Russell problem.** Russell-day log RMSE drops from 1.019 to 0.936 (−0.084). MSCI days improve
   by 0.019. All other event types stay within ±0.003.
3. **Fully separate models overfit, as expected, except on Russell.**
   - Worse on S&P/quad (+0.009), month-end (+0.009) and monthly expiry (+0.008).
   - They also worsen WAPE, which fails the gate.
   - Russell improves by about the same as the specialists. That suggests Russell days really are different
     enough that even 3 days show it.
4. **Caveats.**
   - The Russell gain rests on 3 test days (2022–2024), so it is promising but thin evidence.
   - Bias improves (−13.9% → −11.7%), but WAPE versus the general model is slightly worse (+0.002, not significant).

**Verdict:** `track2_routed_shrunk_v1` is Track 2's candidate. It is the clearest example of
"borrow strength from normal days when events are rare". It should get stronger with more event history:
extending the intraday training back before 2021 would give about 3× the event days.
